# Sorani NER V2 — Tesla T4 training
Keep this notebook, data and Drive folder private. In Colab choose **Runtime → Change runtime type → T4 GPU**. Run sections in order. Full training and frozen test evaluation have separate explicit switches. No full training has been run while preparing this notebook.

Upload only `adyan.txt` and the prepared `sorani-ner-v2-code.zip` to `MyDrive/KurdishNLP/NER-V2/input/`. Open this `.ipynb` with **File → Upload notebook**. A raw-source zip/tar containing exactly one `adyan.txt` is also supported. No alternate dataset is downloaded.
This revision fixes AMP accumulation/overflow validation and enables gradient checkpointing. **T4 capacity is unverified until sections 7–9 pass in your runtime.** Use this notebook with its newly sealed code zip and a new run ID; keep the failed run for diagnosis.


## 1. GPU and runtime verification
Check the actual device before installing anything. Notebook Python may differ from the isolated training Python; unsupported versions use an isolated Python 3.12 without changing the validated model versions.

In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import uuid
from datetime import UTC, datetime
from pathlib import Path

assert Path("/content").is_dir(), "Run this notebook in Google Colab, not on the Mac."
probe = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"],
    text=True,
).strip()
parts = [x.strip() for x in probe.split(",")]
assert len(parts) == 3 and "T4" in parts[0] and int(parts[1]) >= 14000, f"Expected a T4: {probe}"
assert shutil.disk_usage("/content").free >= 30 * 1024**3, (
    "At least 30 GiB of local free disk is needed."
)
print("Notebook Python:", sys.version.split()[0])
print("GPU / VRAM MiB / driver:", probe)
print("Local disk free GiB:", round(shutil.disk_usage("/content").free / 1024**3, 1))

## 2. Google Drive mounting
Only your own Drive is used. Keep at least **12 GiB free** for encoder cache, checkpoint generations and export; Drive's reported free space may not reflect account quota. Active training stays on local Colab disk.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/KurdishNLP/NER-V2")
INPUT = DRIVE_ROOT / "input"
SOURCE_UPLOAD = INPUT / "adyan.txt"  # Or your source archive filename.
CODE_UPLOAD = INPUT / "sorani-ner-v2-code.zip"
for name in ("input", "cache", "runs", "exports"):
    (DRIVE_ROOT / name).mkdir(parents=True, exist_ok=True)
assert SOURCE_UPLOAD.is_file() and CODE_UPLOAD.is_file(), (
    "Upload the two input files to your private input folder."
)
print("Persistent root:", DRIVE_ROOT)

## 3. Repository checkout and revision verification
This verified working-tree snapshot includes the uncommitted NER implementation. A Git clone alone would omit it. Both the base Git commit and exact source-content identity are recorded; this does not misrepresent uncommitted changes as a Git commit. Regenerate the capsule locally with `PYTHONPATH=src .venv/bin/python scripts/prepare_ner_colab.py` if code changes.

In [ ]:
EXPECTED_CODE_SHA256 = "fb88a0c0f63b38b28e384755e940ecf690f1a1aeba19e65b211c345df973b4ee"
EXPECTED_BASE_COMMIT = "756d8942b416fe01e56725723d944730316f09c3"
EXPECTED_CONTENT_SHA256 = "9a32a23ecd00e6aedbe1394d76660d4c56a03c2b6d033dc6242afc3f8285a42f"

In [ ]:
import zipfile

REPO = Path("/content/Kurdish-NLP-stress-fix")


def file_sha(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024**2), b""):
            digest.update(chunk)
    return digest.hexdigest()


assert file_sha(CODE_UPLOAD) == EXPECTED_CODE_SHA256, (
    "Code capsule checksum mismatch. Use the matching notebook/capsule pair."
)
with zipfile.ZipFile(CODE_UPLOAD) as archive:
    manifest = json.loads(archive.read("code-manifest.json"))
    assert manifest["base_git_commit"] == EXPECTED_BASE_COMMIT
    assert manifest["content_sha256"] == EXPECTED_CONTENT_SHA256
    expected = set(manifest["files"]) | {"code-manifest.json"}
    assert set(archive.namelist()) == expected and len(archive.namelist()) == len(expected)
    for name in expected:
        p = Path(name)
        assert not p.is_absolute() and ".." not in p.parts and "\\" not in name
    if not REPO.exists():
        stage = REPO.with_name(".ner-code-" + uuid.uuid4().hex)
        for name in sorted(expected):
            target = stage / name
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(name))
        assert all(file_sha(stage / name) == value for name, value in manifest["files"].items())
        os.replace(stage, REPO)
    assert all(file_sha(REPO / name) == value for name, value in manifest["files"].items()), (
        "Checkout code changed; use a new workspace."
    )
os.chdir(REPO)
print("Base Git commit:", EXPECTED_BASE_COMMIT)
print("Exact working-tree content:", EXPECTED_CONTENT_SHA256)
print("Verified source files:", len(manifest["files"]))

## 4. Dependency installation
Use an isolated environment, not Colab's preinstalled ML packages. The Linux lock preserves spaCy 3.8.7 / spacy-transformers 1.3.9 / transformers 4.49.0 and uses PyTorch 2.6.0+cu124, CuPy 13.4.1, and matching CUDA libraries. No torchvision, torchaudio or unrelated training packages are installed. Kernel restart is unnecessary because all ML work runs in fresh subprocesses. A failed driver probe requires a compatible/fresh GPU runtime.

In [ ]:
VENV = Path("/content/ner-colab-venv")
subprocess.run(
    [
        sys.executable,
        str(REPO / "scripts/ner_colab_environment.py"),
        "--repo",
        str(REPO),
        "--environment",
        str(VENV),
    ],
    check=True,
)
PYTHON = VENV / "bin/python"
ENV = dict(
    os.environ,
    PYTHONPATH=str(REPO / "src"),
    PYTHONHASHSEED="1729",
    TOKENIZERS_PARALLELISM="false",
    OMP_NUM_THREADS="2",
)

# Discover wheel libraries using stdlib only, before any ML process starts.
ENV.update(
    json.loads(
        subprocess.check_output(
            [str(PYTHON), "-m", "kurdish_nlp.ner.cuda_runtime"], cwd=REPO, env=ENV, text=True
        )
    )
)
print("CUDA library search path:", ENV.get("LD_LIBRARY_PATH"))
print("CUDA headers:", ENV.get("CUDA_PATH"))


from contextlib import nullcontext


def run_python(*arguments, log=None, quiet=False):
    with (
        Path(log).open("w") if log else nullcontext() as stream,
        subprocess.Popen(
            [str(PYTHON), *map(str, arguments)],
            cwd=REPO,
            env=ENV,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        ) as process,
    ):
        for line in process.stdout:
            if stream:
                stream.write(line)
                stream.flush()
            if not quiet:
                print(line, end="", flush=True)
        result = process.wait()
    if result:
        raise RuntimeError(
            f"Command failed (exit {result}); inspect {log or 'the output above'}. Do not continue to training."
        )


run_python("-m", "kurdish_nlp.ner.colab", "environment")
# Real CUDA regression tests use tiny randomly initialized XLM-R; no downloads.
run_python("-m", "pytest", "tests/test_ner_colab_numerics.py", "-q", "-k", "cuda")

## 5. Dataset import and integrity verification
Verify the original bytes, then restore the existing local release or rebuild once. A rebuild must reproduce **every** frozen release hash and the 17,396 / 2,058 / 2,145 split. A mismatch blocks training; there is no alternate-data fallback. The 21,599 units are segments, not verified independent articles.

In [ ]:
run_python(
    "-c",
    "from pathlib import Path; from kurdish_nlp.ner.colab_artifacts import import_source,ensure_release; import_source(Path("
    + repr(str(SOURCE_UPLOAD))
    + "),Path('data/ner/raw/adyan.txt')); r=ensure_release(Path('data/ner/raw/adyan.txt'),Path('data/ner/builds/sorani-ner-v2.0'),Path('configs/ner/ner-v2.lock.json'),Path('configs/ner/policy-v2.json')); print('Verified partitions:',r['segments']); print('Entity types:',len(r['labels']))",
)
run_python("-m", "kurdish_nlp.ner.cli", "verify")

## 6. XLM-RoBERTa encoder preparation
Download only the revision and files in `encoder-v2.json`, or restore them from your persistent cache. All file SHA-256 values must match. Keep the cache private. Subsequent training commands use offline loading.

In [ ]:
ENV["HF_HOME"] = str(DRIVE_ROOT / "cache/huggingface")
run_python("scripts/ner_download_encoder.py")
# Copy the verified encoder to local disk; training never reads weights over Drive.
CACHED_ENCODER = Path((REPO / "data/ner/cache/encoder-path.txt").read_text().strip())
ENCODER = Path("/content/ner-encoder")
run_python(
    "-c",
    "from pathlib import Path; import json,shutil; from kurdish_nlp.ner.training import validate_encoder; a=Path("
    + repr(str(CACHED_ENCODER))
    + "); b=Path("
    + repr(str(ENCODER))
    + "); b.mkdir(exist_ok=True); spec=json.loads(Path('configs/ner/encoder-v2.json').read_text()); [(shutil.copyfile(a/n,b/n)) for n in spec['files'] if not (b/n).exists()]; validate_encoder(b); print('Pinned local encoder verified:',spec['revision'])",
)
ENV["HF_HUB_OFFLINE"] = "1"
ENV["TRANSFORMERS_OFFLINE"] = "1"

## 7. CUDA smoke test
Use a new run ID. The T4 retry starts with FP16 (initial scale 128), accumulation 8, evaluation batch 8, and non-reentrant transformer gradient checkpointing. Every record and all 32/24 transformer windows are retained. Checkpointing reduces saved activations for long single records; accumulation only subdivides multi-record batches. This combination still requires a real T4 smoke, stress and benchmark pass.

`numerics.jsonl` records scaled/unscaled named failures, device, NaN/Inf counts, loss scale/backoff, forward window shapes and applied updates. A backward-only AMP overflow discards both components' gradients and replays the same complete batch at a lower scale, at most eight retries. Missing/device errors, non-finite forward/loss/NER gradients, or persistent overflow stop the run. A skipped update cannot pass a preflight.


In [ ]:
RUN_ID = "sorani-ner-v2-t4-" + datetime.now(UTC).strftime("%Y%m%d-%H%M%S")
ACCUMULATE_GRADIENT = 8  # OOM retry with NEW run: 8, 16 or 32; padded batch stays 256.
EVALUATION_BATCH_SIZE = 8  # Reduce for evaluation OOM, then repeat all preflights.
ENABLE_MIXED_PRECISION = True  # FP16 required for this T4 retry.
ENABLE_GRADIENT_CHECKPOINTING = True  # Preserves every record/window.
RUN = Path("/content/ner-runs") / RUN_ID
DRIVE_RUN = DRIVE_ROOT / "runs" / RUN_ID
prepare_args = [
    "-m",
    "kurdish_nlp.ner.colab",
    "prepare",
    "--run",
    RUN,
    "--drive",
    DRIVE_RUN,
    "--encoder",
    ENCODER,
    "--accumulate",
    ACCUMULATE_GRADIENT,
    "--eval-batch",
    EVALUATION_BATCH_SIZE,
]
if ENABLE_MIXED_PRECISION:
    prepare_args.append("--mixed-precision")
if ENABLE_GRADIENT_CHECKPOINTING:
    prepare_args.append("--gradient-checkpointing")
run_python(*prepare_args)
run_python(
    "-m",
    "kurdish_nlp.ner.colab",
    "preflight",
    "--run",
    RUN,
    "--stage",
    "smoke",
    log=RUN / "smoke.log",
    quiet=True,
)
smoke = json.loads((RUN / "preflight-smoke.json").read_text())
print(
    "CUDA smoke:",
    smoke["status"],
    "| Transformer weights changed:",
    smoke["transformer_weight_changed"],
    "| Reload predictions equal:",
    smoke["reload_predictions_equal"],
)
print("Actual prediction payload (engineering smoke only):", smoke["prediction_example"])
print(
    "Checkpoint GiB:",
    round(smoke["checkpoint_bytes"] / 1024**3, 2),
    "| Two full Drive generations GiB:",
    round(smoke["drive_two_generations_bytes"] / 1024**3, 2),
)

## 8. GPU memory stress test
Train on the eight longest training segments under the actual padded batcher and accumulation scheme, The actual training maximum is 819 tokens; the 1,992-token source maximum belongs to TEST and is never used for preflight tuning. A clearly labeled, in-memory repeated-training shape fixture exceeds 1,992 tokens to test a conservative window footprint without adding data or inventing labels. Also test the longest development inputs at the configured evaluation batch size. Do not drop, truncate or repartition records to fit the GPU.
On failure, inspect `stress.log`, `preflight-stress.json` and `numerics.jsonl`; share the failure report and named gradient/scaler attempts. No preflight is marked successful until a finite optimizer update actually applies. A failed run is immutable; use a new run for settings changes.


In [ ]:
run_python(
    "-m",
    "kurdish_nlp.ner.colab",
    "preflight",
    "--run",
    RUN,
    "--stage",
    "stress",
    log=RUN / "stress.log",
    quiet=True,
)
stress = json.loads((RUN / "preflight-stress.json").read_text())
print(
    "Stress:",
    stress["status"],
    "| Longest tested BIO tokens:",
    max(r["bio_tokens"] for r in stress["records"]),
)
print(json.dumps(stress["memory"], indent=2))

## 9. Short GPU throughput benchmark
Measure real CUDA updates using the unchanged batching/optimizer settings, plus one full development inference pass. Estimates include those operations, not Drive I/O or checkpoint writing; early stopping may shorten the run. No CPU benchmark is used for these estimates.

In [ ]:
run_python(
    "-m",
    "kurdish_nlp.ner.colab",
    "preflight",
    "--run",
    RUN,
    "--stage",
    "benchmark",
    log=RUN / "benchmark.log",
    quiet=True,
)
benchmark = json.loads((RUN / "preflight-benchmark.json").read_text())
print("Measured training tokens/s:", round(benchmark["tokens_per_second"], 1))
print("Measured updates/s:", round(benchmark["updates_per_second"], 3))
print("Estimated compute minutes/epoch:", round(benchmark["estimated_epoch_seconds"] / 60, 1))
print(
    "Approximate capped full-run compute hours:",
    round(benchmark["estimated_full_seconds"] / 3600, 2),
)
print("Full dev evaluation seconds:", round(benchmark["full_dev_evaluation_seconds"], 1))
print("Peak allocated VRAM GiB:", round(benchmark["memory"]["peak_allocated_bytes"] / 1024**3, 2))
print(benchmark["estimate_note"])

## 10. Full training configuration
Review every effective change. Learning rate, seed, labels, windows, 15-epoch/20,000-update caps, objective, development selection and patience remain as in the validated config. Passing smoke/stress/benchmark reports are bound to the exact config and runtime-policy hashes. Any config change requires a new run and all preflights.

In [ ]:
run_python("-m", "kurdish_nlp.ner.colab", "gate", "--run", RUN, log=RUN / "gate.log", quiet=True)
print((RUN / "effective-changes.json").read_text())
print("Effective configuration:", RUN / "config.cfg")
print((RUN / "runtime-policy.json").read_text())
print("All three CUDA preflights passed. No full training has started yet.")

## 11. Full training execution
Set the switch only after reviewing section 9's measured resource budget. Watch development F1 and early stopping. Checkpoints back up synchronously after completed development checkpoints (every 200 updates), so Drive I/O adds time. If Drive backup fails, training stops and the last valid backup remains. Keep the notebook connected; Colab runtime limits still apply.

In [ ]:
RUN_FULL_TRAINING = (
    False  # Set True only after all preflights pass and you accept the measured budget.
)
if RUN_FULL_TRAINING:
    run_python("-m", "kurdish_nlp.ner.colab", "train", "--run", RUN, log=RUN / "train.log")
else:
    print(
        "Full training is disabled. Preflight artifacts remain local; no usable model is claimed."
    )

## 12. Checkpoint backup and recovery
Drive stores two complete validated generations, each containing separate best-dev and last checkpoints. The newest generation is published only after copy/checksum verification; `latest.json` and `best.json` point to complete snapshots. Old generations of this same active run are pruned only after successful publication; completed runs are immutable.

**Exact resume is unsupported:** spaCy 3.8.7's standard training loop saves pipeline weights/config, not optimizer/scheduler/RNG state. A recovered interrupted run is for inspection; safely restart from the pinned pretrained encoder under a new run ID and rerun sections 7–11. Do not call `resume_training()` and claim continuity. A recovered completed run can continue to selection/evaluation/export. After reconnecting, rerun sections 1–6 first.

In [ ]:
# Recovery only: fill the previous run ID after reconnecting; leave blank normally.
RECOVER_RUN_ID = ""
if RECOVER_RUN_ID:
    assert RECOVER_RUN_ID.startswith("sorani-ner-v2-") and "/" not in RECOVER_RUN_ID
    RUN_ID = RECOVER_RUN_ID
    DRIVE_RUN = DRIVE_ROOT / "runs" / RUN_ID
    RUN = Path("/content/ner-recovered") / RUN_ID
    run_python(
        "-c",
        "from pathlib import Path; from kurdish_nlp.ner.colab_artifacts import restore_run; restore_run(Path("
        + repr(str(DRIVE_RUN))
        + "),Path("
        + repr(str(RUN))
        + ")); print('Recovery checksums passed')",
    )
    recovered = json.loads((RUN / "run.json").read_text())
    print(
        "Recovered status:", recovered["status"], "| Exact optimizer/scheduler resume: unsupported"
    )
    if recovered["status"] != "trained":
        raise RuntimeError(
            "Interrupted weights recovered for inspection. Start a NEW run from section 7; do not continue to test evaluation."
        )
elif "DRIVE_RUN" in globals() and (DRIVE_RUN / "latest.json").is_file():
    print("Latest valid snapshot:", (DRIVE_RUN / "latest.json").read_text())
else:
    print("No full-training backup yet; preflight models are not production checkpoints.")

## 13. Development-set model selection
Finish all development-only decisions before freezing this checkpoint. Select the existing `model-best` of a completed full run. The original CLI rejects smoke/incomplete runs. The test set has not been predicted on by preflight or training.

In [ ]:
FREEZE_DEVELOPMENT_SELECTION = False  # Set True only when development decisions are finished.
if FREEZE_DEVELOPMENT_SELECTION:
    run_python(
        "-m",
        "kurdish_nlp.ner.cli",
        "select",
        "--model",
        RUN / "checkpoints/model-best",
        "--output",
        RUN / "selection.lock.json",
    )
    run_python(
        "-c",
        "from pathlib import Path; import json; from kurdish_nlp.ner.colab_artifacts import backup_run; r=Path("
        + repr(str(RUN))
        + "); s=json.loads((r/'run.json').read_text()); backup_run(r,Path("
        + repr(str(DRIVE_RUN))
        + "),s['labels'],s['last_completed_eval_step'])",
    )
    selection = json.loads((RUN / "selection.lock.json").read_text())
    print(
        "Frozen development-selected checkpoint. Development performance:",
        selection["dev_performance"],
    )
else:
    print("Selection remains unfrozen; no test evaluation is permitted.")

## 14. Frozen test evaluation
Run once after selection. This uses the existing strict-span evaluator on CPU; speed is explicitly CPU inference speed, not T4 speed. A persistent Drive intent marker prevents accidental retesting after a disconnect. If interrupted after the marker but before a saved report, inspect/recover the run instead of blindly repeating. Never use test results for further selection.

Report rare labels separately. Missing article identities mean this is lexical-duplicate-group-held-out segment evaluation, not document-independent evaluation. Do not compare with historical 88.17 as an equivalent baseline.

In [ ]:
EVALUATE_FROZEN_TEST = False  # Explicit one-time evaluation, only after selection is frozen.
if EVALUATE_FROZEN_TEST:
    assert (RUN / "selection.lock.json").is_file(), "Freeze selection first."
    intent = DRIVE_RUN / "TEST_EVALUATION_STARTED.json"
    if intent.exists():
        raise RuntimeError(
            "Test evaluation was already started for this run. Recover its saved report; do not tune or blindly retest."
        )
    intent.write_text(
        json.dumps(
            {
                "selection_sha256": file_sha(RUN / "selection.lock.json"),
                "started_utc": datetime.now(UTC).isoformat(),
            }
        )
    )
    run_python(
        "-m",
        "kurdish_nlp.ner.cli",
        "evaluate",
        "--model",
        RUN / "checkpoints/model-best",
        "--selection",
        RUN / "selection.lock.json",
        "--output",
        RUN / "test-evaluation.json",
        log=RUN / "evaluation.log",
    )
    # Persist small receipts before the large checkpoint-generation synchronization.
    run_python(
        "-c",
        "from pathlib import Path; import json; from kurdish_nlp.ner.colab_artifacts import atomic_json,backup_run; r=Path("
        + repr(str(RUN))
        + "); d=Path("
        + repr(str(DRIVE_RUN))
        + "); [atomic_json(d/n,json.loads((r/n).read_text())) for n in ['test-evaluation.json','test-evaluation.receipt.json']]; s=json.loads((r/'run.json').read_text()); s['test_predictions_made']=True; atomic_json(r/'run.json',s); backup_run(r,d,s['labels'],s['last_completed_eval_step'])",
    )
    result = json.loads((RUN / "test-evaluation.json").read_text())
    print("FINAL FROZEN TEST — full model; strict exact-span entity scores")
    print(
        json.dumps(
            {
                "micro": result["micro"],
                "per_label": result["per_label"],
                "rare_or_insufficient_support": result["insufficient_support"],
                "span_errors": result["span_errors"],
                "type_confusions": result["overlap_confusion"],
                "inference": {
                    **result["inference"],
                    "mean_latency_ms_per_segment": 1000
                    / result["inference"]["segments_per_second"],
                },
                "model_size_bytes": result["model_size_bytes"],
            },
            indent=2,
        )
    )
    for limitation in result["dataset_limitations"]:
        print("-", limitation)
else:
    print("Frozen test evaluation is disabled; no test scores are claimed.")

## 15. Final model packaging and export
Only an evaluated full model may be exported. The archive includes the complete pipeline, tokenizer/vocabulary, weights, logs, effective config, dependency versions, revision/content identity, dataset lock, selection, evaluation and integrity manifest. It includes no raw corpus. Keep the archive private until redistribution terms are established.

Studio deployment is a later step: verify/extract the archive, install the pinned compatible spaCy runtime, register **`sorani-ner-v2`** in a V2 `TaskModelSpec`, and set `KNLP_NER_MODEL_PATH` to the extracted `sorani-ner-v2/model`. The current registry names V1 explicitly, so changing the path alone would misidentify V2. Keep the current Entities schema, unchanged-text Unicode-code-point offsets and Language ID models. Never deploy a smoke checkpoint.

In [ ]:
EXPORT_EVALUATED_MODEL = False
EXPORT = DRIVE_ROOT / "exports" / f"{RUN_ID}-sorani-ner-v2.tar.gz"
if EXPORT_EVALUATED_MODEL:
    run_python(
        "-m",
        "kurdish_nlp.ner.colab",
        "export",
        "--run",
        RUN,
        "--output",
        EXPORT,
        log=RUN / "export.log",
    )
    completed = DRIVE_RUN / "COMPLETED.json"
    assert not completed.exists(), "Completed run is immutable."
    completed.write_text(
        json.dumps(
            {
                "model_id": "sorani-ner-v2",
                "archive": str(EXPORT),
                "sha256": file_sha(EXPORT),
                "completed_utc": datetime.now(UTC).isoformat(),
            },
            indent=2,
        )
    )
    print("Validated private export:", EXPORT)
    print("Archive SHA-256:", file_sha(EXPORT))
    print("Download from the Drive exports folder; no Studio configuration has been changed.")
else:
    print(
        "Export disabled until full training, development selection and frozen evaluation succeed."
    )